## Setup

In [1]:
import duckdb
import pandas as pd

In [2]:
DATA_PATH = "../data/raw/airline_fraud_3M.parquet"

con = duckdb.connect()

## Dataset Overview

In [3]:
con.sql("""
    SELECT COUNT(*) AS total_transactions
    FROM read_parquet(?)
""", params=[DATA_PATH])

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│            3000000 │
└────────────────────┘

In [7]:
con.sql("""
    SELECT *
    FROM read_parquet(?)
    LIMIT 10
""", params=[DATA_PATH])

┌─────────────────────┬────────────────────┬──────────────┬─────────┬────────────────────────────────┬─────────────────┬──────────────────────────────────┬─────────┬─────────┬───────────────┬──────────┬────────────┬──────────┬─────────────┬─────────────┬─────────────┬──────────────┬────────────────────┬──────────────────┬─────────────────┬──────────┬─────────────────┬──────────────────────┐
│  transaction_date   │     payment_id     │     mrn      │   pnr   │         email_address          │   ip_address    │            device_id             │  route  │ amount  │ amount_in_usd │ currency │ card_type  │ card_bin │ card_last_4 │  bank_name  │ bin_country │ loyalty_tier │ txn_origin_country │ account_age_days │ failed_attempts │ is_fraud │ billing_country │ session_time_seconds │
│      timestamp      │      varchar       │   varchar    │ varchar │            varchar             │     varchar     │             varchar              │ varchar │ double  │    double     │ varchar  │  varchar 

In [6]:
con.sql("""
    DESCRIBE SELECT *
    FROM read_parquet(?)
""", params=[DATA_PATH])

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ transaction_date     │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ payment_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ mrn                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ pnr                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ email_address        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ip_address           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ device_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ route                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ amount               │ DOUBLE      │ YES     │ NUL

The raw dataset contains 3M transactions and 23 columns. DuckDB infers appropriate types for the timestamp, numerical, binary target, and categorical/identifier-like fields. The schema allows NULLs across the columns, but actual missing-value prevalence still needs to be verified.

## Actual NULL Counts

In [4]:
null_summary = con.sql("""
    SUMMARIZE
    SELECT *
    FROM read_parquet(?)
""", params=[DATA_PATH])

null_summary

┌──────────────────────┬─────────────┬──────────────────────────────────┬──────────────────────────────────┬───────────────┬────────────────────────────┬─────────────────────┬───────────────────────────┬────────────────────────────┬────────────────────────────┬─────────┬─────────────────┐
│     column_name      │ column_type │               min                │               max                │ approx_unique │            avg             │         std         │            q25            │            q50             │            q75             │  count  │ null_percentage │
│       varchar        │   varchar   │             varchar              │             varchar              │     int64     │          varchar           │       varchar       │          varchar          │          varchar           │          varchar           │  int64  │  decimal(9,2)   │
├──────────────────────┼─────────────┼──────────────────────────────────┼──────────────────────────────────┼───────────────┼──────

The dataset contains no missing values across all 23 columns. Transactions span from December 1, 2025 to May 29, 2026. The numerical fields have valid-looking non-negative ranges, while several identifier-like fields have high cardinality and will require further investigation before feature selection.

## Exact cardinality + duplicate structure

In [ ]:
identifier_summary = con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT payment_id) AS unique_payment_id,
        COUNT(DISTINCT mrn) AS unique_mrn,
        COUNT(DISTINCT pnr) AS unique_pnr,
        COUNT(DISTINCT email_address) AS unique_email,
        COUNT(DISTINCT ip_address) AS unique_ip,
        COUNT(DISTINCT device_id) AS unique_device,

        ROUND(100 * COUNT(DISTINCT payment_id) / COUNT(*), 2) AS payment_id_unique_pct,
        ROUND(100 * COUNT(DISTINCT mrn) / COUNT(*), 2) AS mrn_unique_pct,
        ROUND(100 * COUNT(DISTINCT pnr) / COUNT(*), 2) AS pnr_unique_pct,
        ROUND(100 * COUNT(DISTINCT email_address) / COUNT(*), 2) AS email_unique_pct,
        ROUND(100 * COUNT(DISTINCT ip_address) / COUNT(*), 2) AS ip_unique_pct,
        ROUND(100 * COUNT(DISTINCT device_id) / COUNT(*), 2) AS device_unique_pct
    FROM read_parquet(?)
""", params=[DATA_PATH])

identifier_summary

┌────────────┬───────────────────┬────────────┬────────────┬──────────────┬───────────┬───────────────┬───────────────────────┬────────────────┬────────────────┬──────────────────┬───────────────┬───────────────────┐
│ total_rows │ unique_payment_id │ unique_mrn │ unique_pnr │ unique_email │ unique_ip │ unique_device │ payment_id_unique_pct │ mrn_unique_pct │ pnr_unique_pct │ email_unique_pct │ ip_unique_pct │ device_unique_pct │
│   int64    │       int64       │   int64    │   int64    │    int64     │   int64   │     int64     │        double         │     double     │     double     │      double      │    double     │      double       │
├────────────┼───────────────────┼────────────┼────────────┼──────────────┼───────────┼───────────────┼───────────────────────┼────────────────┼────────────────┼──────────────────┼───────────────┼───────────────────┤
│    3000000 │           3000000 │    3000000 │    2997852 │       329647 │    463829 │        391306 │                 100.0 │     

## Entity frequency structure

In [8]:
entity_frequency = con.sql("""
    WITH entity_counts AS (
        SELECT
            'email' AS entity_type,
            email_address AS entity_id,
            COUNT(*) AS transaction_count
        FROM read_parquet(?)
        GROUP BY email_address

        UNION ALL

        SELECT
            'ip' AS entity_type,
            ip_address AS entity_id,
            COUNT(*) AS transaction_count
        FROM read_parquet(?)
        GROUP BY ip_address

        UNION ALL

        SELECT
            'device' AS entity_type,
            device_id AS entity_id,
            COUNT(*) AS transaction_count
        FROM read_parquet(?)
        GROUP BY device_id

        UNION ALL

        SELECT
            'pnr' AS entity_type,
            pnr AS entity_id,
            COUNT(*) AS transaction_count
        FROM read_parquet(?)
        GROUP BY pnr
    )

    SELECT
        entity_type,
        COUNT(*) AS unique_entities,
        MIN(transaction_count) AS min_transactions,
        MAX(transaction_count) AS max_transactions,
        ROUND(AVG(transaction_count), 2) AS avg_transactions,
        MEDIAN(transaction_count) AS median_transactions,
        QUANTILE_CONT(transaction_count, 0.75) AS q75_transactions,
        QUANTILE_CONT(transaction_count, 0.90) AS q90_transactions,
        QUANTILE_CONT(transaction_count, 0.95) AS q95_transactions,
        QUANTILE_CONT(transaction_count, 0.99) AS q99_transactions
    FROM entity_counts
    GROUP BY entity_type
    ORDER BY entity_type
""", params=[DATA_PATH, DATA_PATH, DATA_PATH, DATA_PATH])

entity_frequency

┌─────────────┬─────────────────┬──────────────────┬──────────────────┬──────────────────┬─────────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┐
│ entity_type │ unique_entities │ min_transactions │ max_transactions │ avg_transactions │ median_transactions │ q75_transactions │ q90_transactions │ q95_transactions │ q99_transactions │
│   varchar   │      int64      │      int64       │      int64       │      double      │       double        │      double      │      double      │      double      │      double      │
├─────────────┼─────────────────┼──────────────────┼──────────────────┼──────────────────┼─────────────────────┼──────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ device      │          391306 │                1 │             2022 │             7.67 │                 4.0 │              8.0 │             16.0 │             25.0 │             63.0 │
│ email       │          329647 │                1 │   

## Check categorical distributions

In [10]:
categorical_cols = ['currency', 'card_type', 'card_bin', 'bank_name', 'bin_country', 'loyalty_tier', 'txn_origin_country', 'billing_country']

for col in categorical_cols:
    print(f"{col}")

    result = con.sql(f"""
        SELECT
            {col},
            COUNT(*) AS transaction_count,
            ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 2) AS percentage
        FROM read_parquet(?)
        GROUP BY {col}
        ORDER BY transaction_count DESC
    """, params=[DATA_PATH])

    print(result)

currency
┌──────────┬───────────────────┬────────────┐
│ currency │ transaction_count │ percentage │
│ varchar  │       int64       │   double   │
├──────────┼───────────────────┼────────────┤
│ USD      │           1200845 │      40.03 │
│ EUR      │            598714 │      19.96 │
│ JPY      │            300634 │      10.02 │
│ GBP      │            300137 │       10.0 │
│ CAD      │            300011 │       10.0 │
│ AUD      │            150329 │       5.01 │
│ INR      │            149330 │       4.98 │
└──────────┴───────────────────┴────────────┘

card_type
┌────────────┬───────────────────┬────────────┐
│ card_type  │ transaction_count │ percentage │
│  varchar   │       int64       │   double   │
├────────────┼───────────────────┼────────────┤
│ Visa       │           1499670 │      49.99 │
│ MasterCard │           1199952 │       40.0 │
│ Amex       │            300378 │      10.01 │
└────────────┴───────────────────┴────────────┘

card_bin
┌──────────┬───────────────────┬──

### So what have we actually learned?

| Finding                                                        | Interpretation                                                         |
| -------------------------------------------------------------- | ---------------------------------------------------------------------- |
| `loyalty_tier` has 22.2% `None`                                | Missing/undefined loyalty information; meaning needs to be established |
| `bin_country` & `billing_country` have identical distributions | Strong reason to investigate row-level redundancy                      |
| US dominates `bin_country` & `billing_country`                 | Dataset is heavily US-weighted for these fields                        |
| US dominates `txn_origin_country` too                          | But its distribution differs from payment/billing countries            |
| Currency/card/bank distributions are very clean                | Likely synthetic/controlled data generation                            |
| `card_bin` has 24 categories                                   | Low enough to inspect directly; no immediate cardinality problem       |
| No category appears absurdly rare                              | Nothing here screams "bad category" yet                                |

### Test `bin_country` vs `billing_country`

In [11]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN bin_country = billing_country THEN 1 ELSE 0 END)
            AS matching_country_transactions,
        ROUND(
            SUM(CASE WHEN bin_country = billing_country THEN 1 ELSE 0 END)
            * 100 / COUNT(*), 2
        ) AS matching_percentage
    FROM read_parquet(?)
""", params=[DATA_PATH])

┌────────────────────┬───────────────────────────────┬─────────────────────┐
│ total_transactions │ matching_country_transactions │ matching_percentage │
│       int64        │            int128             │       double        │
├────────────────────┼───────────────────────────────┼─────────────────────┤
│            3000000 │                       3000000 │               100.0 │
└────────────────────┴───────────────────────────────┴─────────────────────┘

`bin_country` and `billing_country` are identical for every single transaction. It's perfectly redundant at the row level. 

> `bin_country == billing_country` for 100% of transactions.

In [13]:
for col in categorical_cols:
    print(col)

    result = con.sql(f"""
        SELECT
            {col},
            COUNT(*) AS transactions,
            SUM(is_fraud) AS fraud_transactions,
            ROUND(100 * SUM(is_fraud) / COUNT(*), 3) AS fraud_rate_pct
        FROM read_parquet(?)
        GROUP BY {col}
        ORDER BY transactions DESC
    """, params=[DATA_PATH])

    print(result)

currency
┌──────────┬──────────────┬────────────────────┬────────────────┐
│ currency │ transactions │ fraud_transactions │ fraud_rate_pct │
│ varchar  │    int64     │       int128       │     double     │
├──────────┼──────────────┼────────────────────┼────────────────┤
│ USD      │      1200845 │              17963 │          1.496 │
│ EUR      │       598714 │               8799 │           1.47 │
│ JPY      │       300634 │               4492 │          1.494 │
│ GBP      │       300137 │               4560 │          1.519 │
│ CAD      │       300011 │               4569 │          1.523 │
│ AUD      │       150329 │               2308 │          1.535 │
│ INR      │       149330 │               2309 │          1.546 │
└──────────┴──────────────┴────────────────────┴────────────────┘

card_type
┌────────────┬──────────────┬────────────────────┬────────────────┐
│ card_type  │ transactions │ fraud_transactions │ fraud_rate_pct │
│  varchar   │    int64     │       int128       │  

### Investigate the suspicious country pattern

In [14]:
suspicious_countries = ["EG", "RU", "CN", "BR", "NG"]

result = con.sql("""
    SELECT
        txn_origin_country,
        COUNT(*) AS transactions,
        SUM(is_fraud) AS fraud_transactions,
        COUNT(*) - SUM(is_fraud) AS non_fraud_transactions,
        MIN(transaction_date) AS first_transaction,
        MAX(transaction_date) AS last_transaction,
        COUNT(DISTINCT currency) AS currencies,
        COUNT(DISTINCT card_type) AS card_types,
        COUNT(DISTINCT bank_name) AS banks,
        COUNT(DISTINCT card_bin) AS card_bins
    FROM read_parquet(?)
    WHERE txn_origin_country IN (SELECT UNNEST(?))
    GROUP BY txn_origin_country
    ORDER BY txn_origin_country
""", params=[DATA_PATH, suspicious_countries])

print(result)

┌────────────────────┬──────────────┬────────────────────┬────────────────────────┬─────────────────────┬─────────────────────┬────────────┬────────────┬───────┬───────────┐
│ txn_origin_country │ transactions │ fraud_transactions │ non_fraud_transactions │  first_transaction  │  last_transaction   │ currencies │ card_types │ banks │ card_bins │
│      varchar       │    int64     │       int128       │         int128         │      timestamp      │      timestamp      │   int64    │   int64    │ int64 │   int64   │
├────────────────────┼──────────────┼────────────────────┼────────────────────────┼─────────────────────┼─────────────────────┼────────────┼────────────┼───────┼───────────┤
│ BR                 │         4952 │               4952 │                      0 │ 2025-12-01 00:27:24 │ 2026-05-29 22:37:59 │          7 │          3 │     8 │        24 │
│ CN                 │         5046 │               5046 │                      0 │ 2025-12-01 00:02:13 │ 2026-05-29 23:13:35 │   

## Numerical-variable inspection

In [16]:
numeric_cols = ["amount", "amount_in_usd", "account_age_days", "failed_attempts", "session_time_seconds"]

for col in numeric_cols:
    print(f"\n{col}")

    result = con.sql(f"""
        SELECT
            COUNT(*) AS transactions,
            MIN({col}) AS min_value,
            ROUND(AVG({col}), 3) AS mean,
            ROUND(MEDIAN({col}), 3) AS median,
            ROUND(STDDEV({col}), 3) AS std,
            QUANTILE_CONT({col}, 0.01) AS q01,
            QUANTILE_CONT({col}, 0.25) AS q25,
            QUANTILE_CONT({col}, 0.75) AS q75,
            QUANTILE_CONT({col}, 0.99) AS q99,
            MAX({col}) AS max_value
        FROM read_parquet(?)
    """, params=[DATA_PATH])

    print(result)


amount
┌──────────────┬───────────┬─────────┬────────┬────────┬────────┬────────┬─────────┬────────────────────┬───────────┐
│ transactions │ min_value │  mean   │ median │  std   │  q01   │  q25   │   q75   │        q99         │ max_value │
│    int64     │  double   │ double  │ double │ double │ double │ double │ double  │       double       │  double   │
├──────────────┼───────────┼─────────┼────────┼────────┼────────┼────────┼─────────┼────────────────────┼───────────┤
│      3000000 │       1.0 │ 942.615 │ 771.45 │ 752.45 │  62.43 │ 418.27 │ 1258.77 │ 3537.2702999999933 │  20869.22 │
└──────────────┴───────────┴─────────┴────────┴────────┴────────┴────────┴─────────┴────────────────────┴───────────┘


amount_in_usd
┌──────────────┬───────────┬─────────┬────────┬────────┬────────┬────────┬─────────┬────────────────────┬───────────┐
│ transactions │ min_value │  mean   │ median │  std   │  q01   │  q25   │   q75   │        q99         │ max_value │
│    int64     │  double   │ dou

### Fraud/Non-fraud summary

In [17]:
fraud_numeric_summary = con.sql("""
    SELECT
        is_fraud,

        COUNT(*) AS transactions,

        ROUND(AVG(amount), 2) AS avg_amount,
        ROUND(MEDIAN(amount), 2) AS median_amount,

        ROUND(AVG(amount_in_usd), 2) AS avg_amount_usd,
        ROUND(MEDIAN(amount_in_usd), 2) AS median_amount_usd,

        ROUND(AVG(account_age_days), 2) AS avg_account_age,
        ROUND(MEDIAN(account_age_days), 2) AS median_account_age,

        ROUND(AVG(failed_attempts), 3) AS avg_failed_attempts,
        ROUND(MEDIAN(failed_attempts), 2) AS median_failed_attempts,

        ROUND(AVG(session_time_seconds), 2) AS avg_session_time,
        ROUND(MEDIAN(session_time_seconds), 2) AS median_session_time

    FROM read_parquet(?)
    GROUP BY is_fraud
    ORDER BY is_fraud
""", params=[DATA_PATH])

fraud_numeric_summary

┌──────────┬──────────────┬────────────┬───────────────┬────────────────┬───────────────────┬─────────────────┬────────────────────┬─────────────────────┬────────────────────────┬──────────────────┬─────────────────────┐
│ is_fraud │ transactions │ avg_amount │ median_amount │ avg_amount_usd │ median_amount_usd │ avg_account_age │ median_account_age │ avg_failed_attempts │ median_failed_attempts │ avg_session_time │ median_session_time │
│  int64   │    int64     │   double   │    double     │     double     │      double       │     double      │       double       │       double        │         double         │      double      │       double        │
├──────────┼──────────────┼────────────┼───────────────┼────────────────┼───────────────────┼─────────────────┼────────────────────┼─────────────────────┼────────────────────────┼──────────────────┼─────────────────────┤
│        0 │      2955000 │     935.41 │        771.21 │         810.61 │            635.89 │          358.89 │     

## Date/Time Coverage

In [18]:
date_summary = con.sql("""
    SELECT
        MIN(transaction_date) AS start_date,
        MAX(transaction_date) AS end_date,
        COUNT(DISTINCT CAST(transaction_date AS DATE)) AS unique_dates,
        COUNT(DISTINCT STRFTIME(CAST(transaction_date AS DATE), '%Y-%m')) AS unique_months
    FROM read_parquet(?)
""", params=[DATA_PATH])

date_summary

┌─────────────────────┬─────────────────────┬──────────────┬───────────────┐
│     start_date      │      end_date       │ unique_dates │ unique_months │
│      timestamp      │      timestamp      │    int64     │     int64     │
├─────────────────────┼─────────────────────┼──────────────┼───────────────┤
│ 2025-12-01 00:00:04 │ 2026-05-29 23:59:57 │          180 │             6 │
└─────────────────────┴─────────────────────┴──────────────┴───────────────┘

### Gaps in the dates

In [19]:
date_gap_check = con.sql("""
    WITH date_range AS (
        SELECT
            MIN(CAST(transaction_date AS DATE)) AS start_date,
            MAX(CAST(transaction_date AS DATE)) AS end_date
        FROM read_parquet(?)
    ),
    expected_dates AS (
        SELECT date
        FROM date_range,
        generate_series(start_date, end_date, INTERVAL 1 DAY) AS t(date)
    ),
    actual_dates AS (
        SELECT DISTINCT
            CAST(transaction_date AS DATE) AS date
        FROM read_parquet(?)
    )
    SELECT
        COUNT(*) AS expected_dates,
        COUNT(actual_dates.date) AS observed_dates,
        COUNT(*) - COUNT(actual_dates.date) AS missing_dates
    FROM expected_dates
    LEFT JOIN actual_dates
        ON expected_dates.date = actual_dates.date
""", params=[DATA_PATH, DATA_PATH])

date_gap_check

┌────────────────┬────────────────┬───────────────┐
│ expected_dates │ observed_dates │ missing_dates │
│     int64      │     int64      │     int64     │
├────────────────┼────────────────┼───────────────┤
│            180 │            180 │             0 │
└────────────────┴────────────────┴───────────────┘

## Target Balance and Fraud Distribution over Time

In [20]:
monthly_fraud = con.sql("""
    SELECT
        DATE_TRUNC('month', transaction_date) AS month,
        COUNT(*) AS transactions,
        SUM(is_fraud) AS fraud_transactions,
        ROUND(100 * SUM(is_fraud) / COUNT(*), 3) AS fraud_rate_pct
    FROM read_parquet(?)
    GROUP BY month
    ORDER BY month
""", params=[DATA_PATH])

monthly_fraud

┌─────────────────────┬──────────────┬────────────────────┬────────────────┐
│        month        │ transactions │ fraud_transactions │ fraud_rate_pct │
│      timestamp      │    int64     │       int128       │     double     │
├─────────────────────┼──────────────┼────────────────────┼────────────────┤
│ 2025-12-01 00:00:00 │       517086 │               8332 │          1.611 │
│ 2026-01-01 00:00:00 │       516504 │               7810 │          1.512 │
│ 2026-02-01 00:00:00 │       467318 │               6816 │          1.459 │
│ 2026-03-01 00:00:00 │       516061 │               7355 │          1.425 │
│ 2026-04-01 00:00:00 │       499421 │               7456 │          1.493 │
│ 2026-05-01 00:00:00 │       483610 │               7231 │          1.495 │
└─────────────────────┴──────────────┴────────────────────┴────────────────┘

Fraud prevalence is relatively stable across the six-month period, with December showing the highest observed rate and March the lowest. No strong temporal trend is evident.